In [ ]:
import os
import h5py
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

from libero.force.modules import MujocoSensorReader, WrenchObsWrapper

from robokit.debug_utils.printer import print_batch
from robokit.debug_utils.images import (
    plot_action_wrt_time, save_frames_as_video, plot_force_sensor_wrt_time, concatenate_rgb_images
)

In [ ]:
dataset_root = "/home/Anonymous/code/LIBERO-FT/libero/datasets/"
dataset_subname = "libero_90"
hdf5_fn = "KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"
hdf5_path = os.path.join(dataset_root, dataset_subname, hdf5_fn)

with h5py.File(hdf5_path, "r") as f:
    # attributes
    print(list(f["data"].attrs.keys()))
    print(f["data"].attrs["bddl_file_name"])
    print(os.system("pwd"))
    print(os.path.exists(f["data"].attrs["bddl_file_name"]))
    # print(f["data"].attrs["env_args"])
    hdf5_env_meta = json.loads(f["data"].attrs["env_args"])
    hdf5_env_kwargs = hdf5_env_meta["env_kwargs"]
    print(hdf5_env_kwargs)

    # 顶层 keys
    print("Top-level keys:", list(f.keys()))

    # 递归打印整个树结构（group / dataset）
    def print_h5(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DSET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GRP ] {name}")

    f.visititems(print_h5)


In [ ]:
# build env from task
task_suite = benchmark.get_benchmark_dict()[dataset_subname]()
task_id = 0
task = task_suite.get_task(0)
"""
Task(name='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet', language='close the bottom drawer of the cabinet', problem='Libero', problem_folder='libero_90', bddl_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.bddl', init_states_file='KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet.pruned_init')
"""
while task.name not in hdf5_fn.replace(".hdf5", ""):
    task = task_suite.get_task(task_id)
    task_id += 1
print(task)
task_bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

hdf5_env_kwargs.update({
    "bddl_file_name": task_bddl,
    "camera_heights": 128,
    "camera_widths": 128,
})
if "controller_configs" in hdf5_env_kwargs:
    del hdf5_env_kwargs["controller_configs"]
base_env = OffScreenRenderEnv(**hdf5_env_kwargs)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

env.env.robots[0].controller.reset_goal()

# vis_images = []
# vis_forces = []
# dummy_action = [0.] * 7
# dummy_action[0] = 0.2  # move forward
# dummy_action[1] = -0.2  # move left
# dummy_action[2] = -0.25  # move down
# for step in tqdm(range(300)):
#     obs, reward, done, info = env.step(dummy_action)
#     if step == 70:
#         dummy_action[0] = -0.2   # moving backward
#         dummy_action[1] = 0.7  # move right
#         dummy_action[2] = 0.   # stop moving down
#     elif step == 130:
#         dummy_action[1] = -0.3  # move left
#         dummy_action[2] = -0.2  # move down
#     resized_image = np.array(
#         Image.fromarray(obs["agentview_image"]).resize((512, 512))
#     )
#     vis_images.append(np.flipud(resized_image))
#     vis_forces.append(obs["wrench_ee"])
#
# force_frames, _, _ = plot_force_sensor_wrt_time(
#     np.array(vis_forces)
# )
# combined_vis_images = [
#     concatenate_rgb_images(img, force_img, resize_ratio=1.)
#     for img, force_img in zip(vis_images, force_frames)
# ]
# save_frames_as_video(combined_vis_images, "tmp_images.mp4", fps=10)

env.close()

Replay states from the hdf5 file (Option 1: forward state directly)

In [ ]:
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

def interp_states(s0, s1, nq, nv, K):
    # 假设 flatten = [qpos (nq), qvel (nv), ...rest]
    qpos0, qvel0 = s0[:nq], s0[nq:nq+nv]
    qpos1, qvel1 = s1[:nq], s1[nq:nq+nv]
    rest0 = s0[nq+nv:]
    rest1 = s1[nq+nv:]

    # rest 通常可以直接用 s0 的（或者也插值）
    out = []
    for k in range(1, K+1):
        a = k / (K+1)
        qpos = (1-a)*qpos0 + a*qpos1
        qvel = (1-a)*qvel0 + a*qvel1
        rest = (1-a)*rest0 + a*rest1 if rest0.size == rest1.size else rest0
        out.append(np.concatenate([qpos, qvel, rest], axis=0))
    return out


vis_images = []

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    for iter_idx, demo_key in enumerate(tqdm(list(fin["data"].keys()))):  # e.g., "demo_0", "demo_1", ...
        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 states
        if "states" not in g_in:
            raise KeyError(f"{demo_key} has no 'states' dataset; cannot do state-replay.")
        states = np.array(g_in["states"], dtype=np.float32)  # (T, state_dim), e.g., (251, 51)
        obs = g_in["obs"]
        print("Original obs keys:", list(obs.keys()))
        print("Original obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (217, 128, 128, 3)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_images = resized_images

        # replay -> wrench
        K = 5
        Tlen = states.shape[0]
        wrenches = np.zeros((Tlen, 6), np.float32)
        for t in range(Tlen):
            ## Option 1. Very direct way: set flattened state
            env.set_flattened_state_and_forward(states[t])
            wrenches[t] = env.read_wrench_from_sim()

            ## Option 2. Interp states for smoother sim stepping
            # ws = []
            #
            # env.set_flattened_state_and_forward(states[t])
            # ws.append(env.read_wrench_from_sim())
            #
            # if t < Tlen - 1:
            #     mids = interp_states(states[t], states[t+1], nq, nv, K)
            #     for smid in mids:
            #         env.set_flattened_state_and_forward(smid)
            #         ws.append(env.read_wrench_from_sim())
            #
            # wrenches[t] = np.median(np.stack(ws, axis=0), axis=0)

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 1:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

Replay actions from the hdf5 file (Option 2: step with actions)

In [ ]:
# base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
print(hdf5_env_kwargs)
if "controller_configs" in hdf5_env_kwargs:
    del hdf5_env_kwargs["controller_configs"]
hdf5_env_kwargs["bddl_file_name"] = hdf5_env_kwargs["bddl_file_name"].replace(
    "chiliocosm/bddl_files/libero_100/",
    "/home/Anonymous/code/LIBERO-FT/libero/libero/bddl_files/libero_90/"
)
print("[DEBUG] Using bddl file:", hdf5_env_kwargs["bddl_file_name"])
base_env = OffScreenRenderEnv(**hdf5_env_kwargs)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(5)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])  # len(init_states)=50

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    demo_keys = sorted(list(fin["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys)):  # e.g., "demo_0", "demo_1", ...
        vis_hdf5_images = []
        vis_replay_images = []

        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 actions
        if "actions" not in g_in:
            raise KeyError(f"{demo_key} has no 'actions' dataset; cannot do actions-replay.")
        states = np.array(g_in["states"], dtype=np.float64)  # (T, state_dim), e.g., (251, 51)
        actions = np.array(g_in["actions"], dtype=np.float64)  # (T, action_dim), e.g., (251, 7)
        obs = g_in["obs"]
        # print("HDF5 obs keys:", list(obs.keys()))
        # print("HDF5 obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (251, 128, 128, 3)
        # print("HDF5 obs['actions'] shape:", actions.shape)
        # print("HDF5 obs['states'] shape:", states.shape)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_hdf5_images = resized_images

        # replay -> wrench
        # In HDF5:
        #    s1 s2 ... s9 s10
        # a0 a1 a2 ... a9
        # In Replay:
        #    s1 s2 ... s9 s10
        #    a1 a2 ... a9
        Tlen = actions.shape[0]
        wrenches = np.zeros((Tlen + 1, 6), np.float32)
        env.reset()
        env.set_flattened_state_and_forward(states[0])  # set as the 1st state of hdf5 demo

        # run 10 zero-action steps to stabilize the sim
        # for _ in range(40):
        #     obs, reward, done, info = env.step(np.zeros_like(actions[0]))
        obs = env.env.regenerate_obs_from_state(states[0])
        # wrenches[0] = obs["wrench_ee"]
        vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))

        action_shift = 1  # to align with states
        reset_every = 20000  # reset sim state every N steps to reduce compounding errors
        for t in range(action_shift, Tlen):
            if t % reset_every == (reset_every - 1):
                ## Option 1. Very direct way: set flattened state
                env.set_flattened_state_and_forward(states[t - action_shift])
                obs, reward, done, info = env.step(actions[t] * 0.)
                sim_state = env.env.get_sim_state()
                wrenches[t + 1] = env.read_wrench_from_sim()
            else:
                if t % 5 == 4:
                    env.set_flattened_state_and_forward(states[t - action_shift])
                ## Option 2. Step with actions
                obs, reward, done, info = env.step(actions[t])
                sim_state = env.env.get_sim_state()
                wrenches[t + 1] = obs["wrench_ee"]

            vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))
        success = env.env.check_success()
        print(f"[{demo_key}] success={success}, vis_hdf5 len={len(vis_hdf5_images)}, vis_replay len={len(vis_replay_images)},"
              f" wrench shape={wrenches.shape}")
        # assert len(vis_hdf5_images) == len(vis_replay_images), "Length mismatch between hdf5 images and replay images!"

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 8:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            vis_images = [
                concatenate_rgb_images(hdf5_image, replay_image, resize_ratio=1.)
                for hdf5_image, replay_image in zip(vis_hdf5_images, vis_replay_images)
            ]
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_action_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

Check saved wrench data

In [ ]:
dataset_root = "/home/Anonymous/code/LIBERO-FT/libero/datasets/"
dataset_subname = "libero_90"
hdf5_fn = "KITCHEN_SCENE4_close_the_bottom_drawer_of_the_cabinet_and_open_the_top_drawer_demo.hdf5"
# hdf5_path = os.path.join(dataset_root, dataset_subname, hdf5_fn)
hdf5_path = "tmp_replayed_wrench.hdf5"

def summarize_array(x: np.ndarray):
    """
    x: shape (N, D)
    returns dict of stats per dim
    """
    return {
        "mean": np.mean(x, axis=0),
        "max": np.max(x, axis=0),
        "p01": np.percentile(x, 1, axis=0),
        "p99": np.percentile(x, 99, axis=0),
    }

with h5py.File(hdf5_path, "r") as f:
    # 顶层 keys
    print("Top-level keys:", list(f.keys()))

    # 递归打印整个树结构（group / dataset）
    def print_h5(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DSET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GRP ] {name}")

    f.visititems(print_h5)

    demos = sorted(f["data"].keys())  # ["demo_0", "demo_1", ...]
    all_w = []

    for dk in demos:
        w = np.array(f["data"][dk]["wrenches"], dtype=np.float32)  # (T,6)
        all_w.append(w)

    W = np.concatenate(all_w, axis=0)  # (sum_T, 6)

    # save images
    demo_keys = sorted(list(f["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys, desc="Saving videos")):  # e.g., "demo_0", "demo_1", ...
        obs = f["data"][demo_key]["obs"]
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        save_frames_as_video(resized_images, f"saved_replay_action_images_{iter_idx:02d}.mp4", fps=20)

# 分别统计每个维度
stats_w = summarize_array(W)

# 额外统计模长（更有物理意义）
F = W[:, 0:3]
M = W[:, 3:6]
stats_Fnorm = summarize_array(np.linalg.norm(F, axis=1, keepdims=True))  # (N,1)
stats_Mnorm = summarize_array(np.linalg.norm(M, axis=1, keepdims=True))  # (N,1)
stats_Wnorm = summarize_array(np.linalg.norm(W, axis=1, keepdims=True))  # (N,1)

names = ["Fx","Fy","Fz","Mx","My","Mz"]

print("=== Per-dimension wrench stats over ALL demos ===")
for i, n in enumerate(names):
    print(
        f"{n:>2}  mean={stats_w['mean'][i]:9.3f}  "
        f"max={stats_w['max'][i]:9.3f}  "
        f"p01={stats_w['p01'][i]:9.3f}  "
        f"p99={stats_w['p99'][i]:9.3f}"
    )

print("\n=== Norm stats (scalar) ===")
print(f"||F|| mean={stats_Fnorm['mean'][0]:.3f} max={stats_Fnorm['max'][0]:.3f} p01={stats_Fnorm['p01'][0]:.3f} p99={stats_Fnorm['p99'][0]:.3f}")
print(f"||M|| mean={stats_Mnorm['mean'][0]:.3f} max={stats_Mnorm['max'][0]:.3f} p01={stats_Mnorm['p01'][0]:.3f} p99={stats_Mnorm['p99'][0]:.3f}")
print(f"||W|| mean={stats_Wnorm['mean'][0]:.3f} max={stats_Wnorm['max'][0]:.3f} p01={stats_Wnorm['p01'][0]:.3f} p99={stats_Wnorm['p99'][0]:.3f}")

Fn = np.linalg.norm(W[:, :3], axis=1)
print("contact ratio (Fn>1N):", (Fn>1).mean())
print("contact ratio (Fn>10N):", (Fn>10).mean())
print("percentiles:", np.percentile(Fn, [50, 90, 95, 99, 99.5, 99.9]).astype(np.int64))

# per_demo_max = []
# for each demo:
#     per_demo_max.append(np.linalg.norm(w[:, :3], axis=1).max())
# print(np.percentile(per_demo_max, [50, 90, 95, 99]))

In [ ]:
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])  # len(init_states)=50

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv

demo_h5_path = hdf5_path
out_h5_path = "tmp_replayed_wrench.hdf5"
with h5py.File(demo_h5_path, "r") as fin, h5py.File(out_h5_path, "w") as fout:
    # 复制全局 attrs（如果有）
    num_out_attrs = 0
    for k, v in fin.attrs.items():
        fout.attrs[k] = v
        num_out_attrs += 1
    print(f"Copied {num_out_attrs} global attributes from input to output HDF5")

    print("Demo keys:", list(fin["data"].keys()))
    fout.create_group("data")

    demo_keys = sorted(list(fin["data"].keys()), key=lambda x: int(x.split("_")[1]))
    for iter_idx, demo_key in enumerate(tqdm(demo_keys)):  # e.g., "demo_0", "demo_1", ...
        vis_hdf5_images = []
        vis_replay_images = []

        print("Processing demo group:", demo_key)
        g_in = fin["data"][demo_key]
        g_out = fout["data"].create_group(demo_key)

        # 复制原有数据集（不破坏结构）
        print("Copying dataset...")
        for k in g_in.keys():  # k in: actions, dones, obs, rewards, robot_states, states
            g_in.copy(k, g_out)
            pass

        # 读取 actions
        if "actions" not in g_in:
            raise KeyError(f"{demo_key} has no 'actions' dataset; cannot do actions-replay.")
        states = np.array(g_in["states"], dtype=np.float32)  # (T, state_dim), e.g., (251, 51)
        actions = np.array(g_in["actions"], dtype=np.float32)  # (T, action_dim), e.g., (251, 7)
        robot_states = np.array(g_in["robot_states"], dtype=np.float32)
        ee_states = np.array(g_in["obs"]["ee_states"], dtype=np.float32)
        ee_pos = np.array(g_in["obs"]["ee_pos"], dtype=np.float32)
        ee_ori = np.array(g_in["obs"]["ee_ori"], dtype=np.float32)
        gripper_states = np.array(g_in["obs"]["gripper_states"], dtype=np.float32)
        print("robot_states:", robot_states[0])
        print("ee_states:", ee_states[0])
        print("ee_pos:", ee_pos[0])
        print("ee_ori:", ee_ori[0])
        print("gripper_states:", gripper_states[0])
        obs = g_in["obs"]
        # print("HDF5 obs keys:", list(obs.keys()))
        # print("HDF5 obs['agentview_rgb'] shape:", obs["agentview_rgb"].shape)  # (T,H,W,C), e.g., (251, 128, 128, 3)
        # print("HDF5 obs['actions'] shape:", actions.shape)
        # print("HDF5 obs['states'] shape:", states.shape)
        resized_images = [np.flipud(np.array(Image.fromarray(x).resize((512, 512)))) for x in obs["agentview_rgb"]]
        vis_hdf5_images = resized_images

        # replay -> wrench
        # In HDF5:
        #    s1 s2 ... s9 s10
        # a0 a1 a2 ... a9
        # In Replay:
        #    s1 s2 ... s9 s10
        #    a1 a2 ... a9
        Tlen = actions.shape[0]
        wrenches = np.zeros((Tlen + 1, 6), np.float32)
        env.reset()
        env.set_flattened_state_and_forward(states[0])  # set as the 1st state of hdf5 demo

        # run 10 zero-action steps to stabilize the sim
        for _ in range(10):
            obs, reward, done, info = env.step(np.zeros_like(actions[0]))
        sim_state = env.env.get_sim_state()
        wrenches[0] = obs["wrench_ee"]
        print("obs.robot0_eef_quat:", obs["robot0_eef_quat"])
        # print_batch("obs", obs)
        # print_batch("reward", reward)
        break
        vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))

        action_shift = 0  # to align with states
        for t in range(action_shift, Tlen):
            ## Option 2. Step with actions
            obs, reward, done, info = env.step(actions[t])
            wrenches[t + 1] = obs["wrench_ee"]
            vis_replay_images.append(np.flipud(np.array(Image.fromarray(obs["agentview_image"]).resize((512, 512)))))
        success = env.env.check_success()
        print(f"[{demo_key}] success={success}, vis_hdf5 len={len(vis_hdf5_images)}, vis_replay len={len(vis_replay_images)},"
              f" wrench shape={wrenches.shape}")
        # assert len(vis_hdf5_images) == len(vis_replay_images), "Length mismatch between hdf5 images and replay images!"

        # 新增 dataset
        g_out.create_dataset("wrenches", data=wrenches, dtype="float32")

        if iter_idx == 1:
            force_frames, _, _ = plot_force_sensor_wrt_time(wrenches)
            vis_images = [
                concatenate_rgb_images(hdf5_image, replay_image, resize_ratio=1.)
                for hdf5_image, replay_image in zip(vis_hdf5_images, vis_replay_images)
            ]
            combined_vis_images = [
                concatenate_rgb_images(img, force_img, resize_ratio=1.)
                for img, force_img in zip(vis_images, force_frames)
            ]
            save_frames_as_video(combined_vis_images, f"tmp_replay_action_images.mp4", fps=10)
            # break  # only do one demo for now

env.close()

Check physical parameters in the sim

In [ ]:
from libero.force.physics_helper import PhysicsHelper
base_env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=128, camera_widths=128)
env = WrenchObsWrapper(base_env, force_sensor="gripper0_force_ee", torque_sensor="gripper0_torque_ee")
env.seed(0)
env.reset()

init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

sim = env.env.sim  # robosuite.utils.binding_utils.MjSim
nq, nv = sim.model.nq, sim.model.nv
print(type(sim.model))


''' Find out physical parameters of the env '''
model = env.env.sim.model
phy_helper = PhysicsHelper(
    model,
    object_keywords=("drawer", "cabinet"),
)
phy_helper.apply_dynamics_shift(
)
phy_helper.restore_original_params()

env.close()

In [ ]:
''' quant - euler '''
import robosuite.utils.transform_utils as T
def quat_xyzw_to_rotvec(q_xyzw: np.ndarray) -> np.ndarray:
    """
    输入: quat in xyzw
    输出: rotation vector (axis-angle), shape (3,) = theta * axis
    """
    # 有些 robosuite 版本函数名叫 quat2axisangle
    rotvec = T.quat2axisangle(q_xyzw)   # (3,)
    return rotvec

q_xyzw = np.array([0.9996168, -0.00674171, -0.02033794, -0.0175275 ])
euler_xyz = quat_xyzw_to_rotvec(q_xyzw)
print("euler_xyz (radians):", euler_xyz)

obs_q_xyzw = np.array([9.99617687e-01,  2.46561481e-04, -2.76432318e-02, -5.20345748e-04])
obs_euler_xyz = quat_xyzw_to_rotvec(q_xyzw)
print("obs_euler_xyz (radians):", obs_euler_xyz)